# 01 — Dane

**Cel notebooka:** pobrać z [OpenF1](https://openf1.org/docs) wszystko, czego potrzebujemy do przewidzenia wyniku GP Singapuru 2026, i zapisać to na dysk — kolejne notebooki tylko wczytują pliki CSV.

| Plik | Zawartość | Po co |
|---|---|---|
| `races_2026.csv` | lista wyścigów 2026 z numerem rundy | oś czasu |
| `results_2026.csv` | wyniki wyścigów | **zmienne docelowe** (wygrana, top 3, top 10) + forma |
| `quali_2026.csv` | wyniki kwalifikacji, także Singapur | pozycja startowa, tempo |
| `sprint_2026.csv` | wyniki sprintów, także Singapur | dodatkowa informacja o tempie w weekendzie |
| `fastest_laps_2026.csv` | kto miał najszybsze okrążenie | **zmienna docelowa** nr 4 |
| `teammate_gaps_hist.csv` | różnica w kwalifikacjach do partnera z zespołu, 2023–2025 | **umiejętności kierowcy** niezależne od samochodu |

Każdy krok ma ten sam układ: *co i dlaczego* → kod → *co tu się dzieje*.

> Pierwsze uruchomienie trwa kilka minut (limit zapytań API). Kolejne — sekundy, bo wszystko czytamy z `data/raw`.

## 1.1 Funkcja do pobierania danych

**Co robimy:** jedną funkcję `get()`, przez którą przechodzi każde zapytanie do OpenF1.

**Dlaczego:** będziemy pobierać dane z ponad stu sesji. Bez zapisu na dysk każde ponowne uruchomienie notebooka ściągałoby wszystko od nowa — wolno i obciążając API. Funkcja przy pierwszym wywołaniu pobiera dane i zapisuje JSON, przy kolejnych czyta z pliku.

In [1]:
import json, time
from pathlib import Path
import requests
import pandas as pd

BASE_URL = "https://api.openf1.org/v1"
CACHE_DIR = Path("data/raw")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def get(endpoint, **params):
    """Pobiera dane z OpenF1; odpowiedź zapisuje na dysk i przy kolejnym wywołaniu czyta z pliku."""
    key = endpoint + "_" + "_".join(f"{k}={v}" for k, v in sorted(params.items()))
    path = CACHE_DIR / f"{key}.json"
    if path.exists():
        return pd.DataFrame(json.loads(path.read_text()))
    for attempt in range(5):
        resp = requests.get(f"{BASE_URL}/{endpoint}", params=params, timeout=30)
        if resp.status_code != 429:
            break
        wait = 5 * 2 ** attempt
        print(f"429 (za dużo zapytań) — czekam {wait} s...")
        time.sleep(wait)
    time.sleep(1)
    if resp.status_code == 404:  # OpenF1 zwraca 404 "No results found", gdy danych (jeszcze) nie ma
        return pd.DataFrame()
    resp.raise_for_status()
    data = resp.json()
    path.write_text(json.dumps(data))
    return pd.DataFrame(data)

**Co tu się dzieje:**
- `**params` zbiera argumenty nazwane: `get("sessions", year=2026)` → zapytanie `.../sessions?year=2026`.
- `key` to nazwa pliku z endpointu i parametrów, np. `sessions_year=2026.json`. Parametry są posortowane, więc ta sama kombinacja zawsze trafia do tego samego pliku.
- **Pętla `for attempt`** — API ma limit zapytań. Po przekroczeniu odpowiada kodem **429 Too Many Requests** (przekonaliśmy się przy pierwszym uruchomieniu!). Czekamy i próbujemy ponownie, za każdym razem 2× dłużej: 5, 10, 20, 40, 80 s. To standardowy wzorzec *exponential backoff*.
- **404** — OpenF1 tak sygnalizuje „brak danych” (np. wynik wyścigu, który jeszcze się nie odbył). Zwracamy pustą tabelę i **nie zapisujemy jej na dysk** — inaczej po wyścigu dalej czytalibyśmy pusty plik.
- `raise_for_status()` rzuca błąd przy pozostałych problemach (500 itp.).
- ⚠️ Gdy dane na serwerze się zmienią, trzeba usunąć odpowiedni plik z `data/raw`, żeby pobrać świeżą wersję.

## 1.2 Kalendarz sezonu 2026

**Co robimy:** pobieramy wszystkie sesje 2026, zamieniamy daty z tekstu na daty i wyrzucamy odwołane sesje.

**Dlaczego:** z tej tabeli wiemy, które sesje już się odbyły, i bierzemy z niej `session_key` / `meeting_key` do kolejnych zapytań. Tekstu nie da się porównać z „teraz”, daty — tak.

In [2]:
sessions = get("sessions", year=2026)
for col in ["date_start", "date_end"]:
    sessions[col] = pd.to_datetime(sessions[col])
sessions = sessions[~sessions["is_cancelled"]]

sessions[sessions["country_name"] == "Singapore"][["session_key", "session_name", "date_start"]]

,session_key,session_name,date_start
96,11378,Practice 1,2026-10-09 08:30:00+00:00
97,11379,Sprint Qualifying,2026-10-09 12:30:00+00:00
98,11383,Sprint,2026-10-10 09:00:00+00:00
99,11384,Qualifying,2026-10-10 13:30:00+00:00
100,11388,Race,2026-10-11 12:00:00+00:00


**Co tu się dzieje:**
- `pd.to_datetime` zamienia `"2026-10-11T12:00:00+00:00"` na datę ze strefą UTC.
- `~` to negacja — zostawiamy wiersze, gdzie `is_cancelled` jest fałszem.
- Ostatnie wyrażenie w komórce Jupyter wyświetla jako tabelę.

**Wniosek:** Singapur to **weekend sprinterski** (Practice 1 → Sprint Qualifying → Sprint → Qualifying → Race). Tylko jeden trening, ale za to dodatkowe dane ze sprintu. Wyścig: 12:00 UTC = **14:00 w Polsce**.

## 1.3 Wyścigi: zakończone + dzisiejszy

**Co robimy:** bierzemy wyścigi, które już wystartowały albo startują dziś, numerujemy je jako rundy i oznaczamy, które są zakończone.

**Dlaczego:** zakończone wyścigi to zbiór treningowy (~22 wiersze ze znanym wynikiem na wyścig). Dzisiejszy — Singapur — to ten, dla którego robimy predykcję: znamy jego kwalifikacje i sprint, ale nie wynik. Numer rundy przyda się do liczenia formy („ostatnie 3 wyścigi”).

In [3]:
now = pd.Timestamp.now(tz="UTC")
races = (
    sessions[(sessions["session_name"] == "Race") & (sessions["date_start"] < now + pd.Timedelta(days=1))]
    .sort_values("date_start")
    .reset_index(drop=True)
)
races["round"] = races.index + 1
races["finished"] = races["date_end"] < now
races_done = races[races["finished"]]

TARGET_MEETING = races.loc[races["country_name"] == "Singapore", "meeting_key"].item()

print(len(races_done), "zakończonych wyścigów; cel predykcji: meeting_key =", TARGET_MEETING)
races[["round", "session_key", "meeting_key", "country_name", "location", "date_start", "finished"]]

17 zakończonych wyścigów; cel predykcji: meeting_key = 1296


,round,session_key,meeting_key,country_name,location,date_start,finished
0,1,11234,1279,Australia,Melbourne,2026-03-08 04:00:00+00:00,True
1,2,11245,1280,China,Shanghai,2026-03-15 07:00:00+00:00,True
2,3,11253,1281,Japan,Suzuka,2026-03-29 05:00:00+00:00,True
3,4,11280,1284,United States,Miami Gardens,2026-05-03 17:00:00+00:00,True
4,5,11291,1285,Canada,Montréal,2026-05-24 20:00:00+00:00,True
5,6,11299,1286,Monaco,Monte Carlo,2026-06-07 13:00:00+00:00,True
6,7,11307,1287,Spain,Barcelona,2026-06-14 13:00:00+00:00,True
7,8,11315,1288,Austria,Spielberg,2026-06-28 13:00:00+00:00,True
8,9,11326,1289,United Kingdom,Silverstone,2026-07-05 14:00:00+00:00,True
9,10,11334,1290,Belgium,Spa-Francorchamps,2026-07-19 13:00:00+00:00,True


**Co tu się dzieje:**
- `tz="UTC"` — pandas nie porówna daty ze strefą z datą bez strefy.
- Filtrujemy po `session_name == "Race"`, a nie `session_type`, bo **sprint też ma `session_type == "Race"`**.
- `now + 1 dzień` — dzięki temu łapiemy dzisiejszy wyścig, nawet jeśli jeszcze nie wystartował.
- `TARGET_MEETING` — klucz weekendu w Singapurze; `.item()` wyciąga pojedynczą wartość i rzuci błąd, gdyby pasowało 0 albo 2 wiersze (tania asercja).

**Do zauważenia w danych:**
1. GP Bahrajnu i Arabii Saudyjskiej w kwietniu są odwołane, a „Bahrain” wraca 4 października z `location` = **Kuala Lumpur**. Błąd w danych czy coś, czego nie wiemy? Lekcja: **zawsze oglądaj surowe dane**.
2. W `sessions` są też testy przedsezonowe („Day 1”…) — odpadły dzięki filtrowi na `"Race"`.

## 1.4 Kierowcy weekendu

**Co robimy:** funkcję, która dla danego weekendu (`meeting_key`) zwraca tabelę: numer → skrót nazwiska → zespół.

**Dlaczego:** wszystkie wyniki w OpenF1 znają kierowcę tylko po `driver_number`. A numer **nie jest stałym identyfikatorem** — w 2026 Norris jeździ z #1 (mistrz świata), a Verstappen z #3. Do łączenia danych między sezonami używamy `name_acronym` (VER, NOR…), a `team_name` jest potrzebny do porównań z partnerem z zespołu. Pytamy po `meeting_key` (1 zapytanie na weekend) zamiast po każdej sesji osobno — mniej zapytań, mniej problemów z limitem.

In [4]:
DRIVER_COLS = ["driver_number", "name_acronym", "full_name", "team_name"]

def weekend_drivers(meeting_key):
    """Kierowcy weekendu: driver_number -> name_acronym, team_name."""
    drv = get("drivers", meeting_key=meeting_key)
    return drv[DRIVER_COLS].drop_duplicates("driver_number")

def session_key(meeting_key, session_name):
    """session_key konkretnej sesji weekendu albo None, jeśli takiej sesji nie było (np. sprint)."""
    s = sessions[(sessions["meeting_key"] == meeting_key) & (sessions["session_name"] == session_name)]
    return s["session_key"].item() if len(s) else None

weekend_drivers(TARGET_MEETING)

,driver_number,name_acronym,full_name,team_name
0,1,NOR,Lando NORRIS,McLaren
1,3,VER,Max VERSTAPPEN,Red Bull Racing
2,5,BOR,Gabriel BORTOLETO,Audi
3,6,HAD,Isack HADJAR,Red Bull Racing
4,10,GAS,Pierre GASLY,Alpine
5,11,PER,Sergio PEREZ,Cadillac
6,12,ANT,Kimi ANTONELLI,Mercedes
7,14,ALO,Fernando ALONSO,Aston Martin
8,16,LEC,Charles LECLERC,Ferrari
9,18,STR,Lance STROLL,Aston Martin


**Co tu się dzieje:**
- `drivers?meeting_key=...` zwraca kierowców dla **każdej sesji** weekendu, więc ten sam kierowca występuje kilka razy — `drop_duplicates` zostawia jeden wiersz na numer.
- `session_key()` to pomocnik: „daj mi klucz kwalifikacji z weekendu X”. Zwraca `None`, gdy sesji nie było — weekendy bez sprintu nie mają sesji `"Sprint"`.

## 1.5 Wyniki wyścigów

**Co robimy:** dla każdego zakończonego wyścigu pobieramy `session_result` i doklejamy kierowców.

**Dlaczego:** to nasze **zmienne docelowe** — z pozycji na mecie policzymy później „wygrał / top 3 / top 10”. A historia wyników posłuży też do cech formy.

In [5]:
frames = []
for race in races_done.itertuples():
    res = get("session_result", session_key=race.session_key)
    df = res.merge(weekend_drivers(race.meeting_key), on="driver_number", how="left")
    df["round"] = race.round
    df["meeting_key"] = race.meeting_key
    frames.append(df)

results = pd.concat(frames, ignore_index=True)
print(results.shape)
results.head()

(374, 15)


,position,driver_number,number_of_laps,points,dnf,dns,dsq,duration,gap_to_leader,meeting_key,session_key,name_acronym,full_name,team_name,round
0,1.0,63,58,25.0,False,False,False,4986.801,0,1279,11234,RUS,George RUSSELL,Mercedes,1
1,2.0,12,58,18.0,False,False,False,4989.775,2.974,1279,11234,ANT,Kimi ANTONELLI,Mercedes,1
2,3.0,16,58,15.0,False,False,False,5002.320,15.519,1279,11234,LEC,Charles LECLERC,Ferrari,1
3,4.0,44,58,12.0,False,False,False,5002.945,16.144,1279,11234,HAM,Lewis HAMILTON,Ferrari,1
4,5.0,1,58,10.0,False,False,False,5038.542,51.741,1279,11234,NOR,Lando NORRIS,McLaren,1


**Co tu się dzieje:**
- `itertuples()` iteruje po wierszach — `race.session_key` zamiast `race["session_key"]`.
- `merge(..., how="left")` — zostawia **wszystkie** wiersze wyników, nawet gdyby kierowcy brakowało w `drivers` (wtedy `name_acronym` = NaN). Dzięki temu taki problem zobaczymy, zamiast po cichu zgubić wiersz.
- `pd.concat` skleja 16 małych tabel w jedną.

### Kontrola jakości

**Dlaczego:** błędy w danych przechodzą po cichu przez cały pipeline i wychodzą dopiero jako dziwne predykcje. Taniej złapać je teraz.

In [6]:
print("Brak dopasowania kierowcy:", results["name_acronym"].isna().sum())
print("Kierowców na wyścig:", results.groupby("round").size().unique())
print()
print(results[["dnf", "dns", "dsq"]].sum())
results[results["dnf"]][["round", "name_acronym", "team_name", "position", "number_of_laps"]].head(10)

Brak dopasowania kierowcy: 0
Kierowców na wyścig: [22]

dnf    71
dns     7
dsq     0
dtype: int64


,round,name_acronym,team_name,position,number_of_laps
17,1,ALO,Aston Martin,NaN,21
18,1,BOT,Cadillac,NaN,15
19,1,HAD,Red Bull Racing,NaN,10
37,2,VER,Red Bull Racing,NaN,45
38,2,ALO,Aston Martin,NaN,32
39,2,STR,Aston Martin,NaN,9
64,3,STR,Aston Martin,NaN,30
65,3,BEA,Haas F1 Team,NaN,20
84,4,HUL,Audi,NaN,7
85,4,LAW,Racing Bulls,NaN,6


**Co widać:**
- **~67 DNF i 7 DNS na ~350 startów** — prawie co piąty start kończy się wycofaniem. Nowe jednostki napędowe 2026 się psują → **niezawodność zespołu** będzie ważną cechą.
- Kierowca z DNF czasem ma `position` (sklasyfikowany, bo przejechał ≥90% dystansu), a czasem `NaN`. Co z tym zrobić, zdecydujemy w `03_model`.

## 1.6 Kwalifikacje i sprinty (także Singapur)

**Co robimy:** dla każdego weekendu — łącznie z dzisiejszym — pobieramy wyniki kwalifikacji i sprintu.

**Dlaczego:**
- **Kwalifikacje** = pozycja startowa (najważniejsza pojedyncza cecha, zwłaszcza w Singapurze, gdzie trudno wyprzedzać) + czyste tempo samochodu i kierowcy.
- Endpoint `starting_grid` jest w 2026 pusty, więc **pozycja w kwalifikacjach zastępuje pozycję startową**. Tracimy informację o karach na starcie — świadome uproszczenie.
- **Sprint** — dodatkowa informacja o tempie wyścigowym w tym samym weekendzie (tylko 6 weekendów, w tym Singapur).

W kwalifikacjach `duration` to **lista trzech czasów `[Q1, Q2, Q3]`** (z `None` w segmentach, w których kierowca już nie jechał). Liczymy z niej:
- `best_time` — najlepszy czas kierowcy,
- `gap_to_pole_pct` — o ile procent wolniej od najszybszego. Procenty zamiast sekund, bo okrążenie w Monako i na Spa ma zupełnie inną długość.

In [7]:
def best_time(durations):
    """Najlepszy czas z [Q1, Q2, Q3]; None, gdy kierowca nie ustanowił żadnego czasu."""
    if not isinstance(durations, list):
        return None
    times = [t for t in durations if t is not None]
    return min(times) if times else None

def load_weekend_session(session_name):
    """Wyniki danej sesji (np. 'Qualifying') ze wszystkich weekendów w `races`, z doklejonymi kierowcami."""
    frames = []
    for race in races.itertuples():
        key = session_key(race.meeting_key, session_name)
        if key is None:
            continue
        res = get("session_result", session_key=key)
        if res.empty:
            continue
        df = res.merge(weekend_drivers(race.meeting_key), on="driver_number", how="left")
        df["round"] = race.round
        df["meeting_key"] = race.meeting_key
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

quali = load_weekend_session("Qualifying")
quali["position"] = pd.to_numeric(quali["position"], errors="coerce")
quali["best_time"] = quali["duration"].apply(best_time)
quali["gap_to_pole_pct"] = (quali["best_time"] / quali.groupby("round")["best_time"].transform("min") - 1) * 100

sprint = load_weekend_session("Sprint")

print("Kwalifikacje:", quali["round"].nunique(), "weekendów | sprinty:", sprint["round"].nunique(), "weekendów")
quali[quali["meeting_key"] == TARGET_MEETING][["position", "name_acronym", "team_name", "best_time", "gap_to_pole_pct"]].head(10)

Kwalifikacje: 17 weekendów | sprinty: 6 weekendów


,position,name_acronym,team_name,best_time,gap_to_pole_pct
347,1.0,VER,Red Bull Racing,91.373,0.000000
348,2.0,LEC,Ferrari,91.427,0.059098
349,3.0,HAM,Ferrari,91.496,0.134613
350,4.0,ANT,Mercedes,91.562,0.206844
351,5.0,NOR,McLaren,91.712,0.371007
352,6.0,RUS,Mercedes,91.747,0.409311
353,7.0,PIA,McLaren,91.860,0.532980
354,8.0,GAS,Alpine,92.225,0.932442
355,9.0,LAW,Racing Bulls,92.319,1.035317
356,10.0,LIN,Racing Bulls,92.498,1.231217


**Co tu się dzieje:**
- `pd.to_numeric(..., errors="coerce")` — w kwalifikacjach w Singapurze Hadjar ma pozycję `"RT"` (wycofany) zamiast liczby. Ten jeden tekst zamienia całą kolumnę w tekst i psuje porównania (`"2" < "10"` jest fałszem!). `coerce` zamienia wszystko, co nie jest liczbą, na `NaN` — później potraktujemy to jako start z końca stawki.
- `load_weekend_session` to uogólnienie pętli z 1.5 — ta sama logika dla kwalifikacji i sprintu, więc piszemy ją raz.
- `groupby("round")["best_time"].transform("min")` — dla każdego wiersza czas pole position **jego** weekendu (`transform` zwraca wynik o tej samej długości co tabela, więc można od razu dzielić).
- Na końcu widzimy **dzisiejszą kolejność startową w Singapurze** — to wejście do predykcji.

**Uproszczenie do zapamiętania:** Q3 jest szybsze niż Q1 (tor się „gumuje”), więc `best_time` lekko faworyzuje kierowców z Q3. Dla czołówki to bez znaczenia, dla końca stawki — drobny szum.

## 1.7 Najszybsze okrążenie

**Co robimy:** dla każdego zakończonego wyścigu pobieramy wszystkie okrążenia (`laps`) i szukamy najkrótszego.

**Dlaczego:** `session_result` nie mówi, kto miał najszybsze okrążenie — trzeba to policzyć z okrążeń. To nasza czwarta zmienna docelowa.

In [8]:
fl_rows = []
for race in races_done.itertuples():
    laps = get("laps", session_key=race.session_key)
    best = laps.loc[laps["lap_duration"].idxmin()]
    fl_rows.append({"round": race.round, "driver_number": best["driver_number"],
                    "fastest_lap_time": best["lap_duration"], "lap_number": best["lap_number"]})

fastest_laps = pd.DataFrame(fl_rows).merge(
    results[["round", "driver_number", "name_acronym", "team_name", "position"]],
    on=["round", "driver_number"], how="left",
)
fastest_laps

,round,driver_number,fastest_lap_time,lap_number,name_acronym,team_name,position
0,1,3,82.091,42,VER,Red Bull Racing,6.0
1,2,12,95.275,52,ANT,Mercedes,1.0
2,3,12,92.432,49,ANT,Mercedes,1.0
3,4,1,91.869,35,NOR,McLaren,2.0
4,5,12,74.210,68,ANT,Mercedes,1.0
5,6,12,73.481,76,ANT,Mercedes,1.0
6,7,44,80.122,44,HAM,Ferrari,1.0
7,8,12,70.374,59,ANT,Mercedes,3.0
8,9,12,91.777,37,ANT,Mercedes,15.0
9,10,1,108.890,44,NOR,McLaren,7.0


**Co tu się dzieje:**
- `idxmin()` zwraca indeks wiersza z najmniejszym `lap_duration` (NaN-y — np. okrążenia przerwane czerwoną flagą — są pomijane).
- Okrążeń wyjazdowych z boksu (`is_pit_out_lap`) nie trzeba odfiltrowywać: są wolne, więc i tak nie będą minimum.
- Doklejamy `position` — zobaczysz, że najszybsze okrążenie **często nie należy do zwycięzcy**. To zapowiedź, że ten cel będzie najtrudniejszy.

## 1.8 Umiejętności kierowców: porównanie z partnerem z zespołu (2023–2025)

**Co robimy:** dla każdych kwalifikacji z lat 2023–2025 liczymy, o ile procent kierowca był szybszy/wolniejszy od partnera z zespołu.

**Dlaczego:** wynik kierowcy to mieszanka „kierowca + samochód”. Partner z zespołu ma **ten sam samochód**, więc różnica do niego to w dużej mierze umiejętności. Przepisy 2026 zmieniły samochody, ale nie kierowców — dlatego ta informacja przenosi się między epokami, a dane o samochodach (niezawodność, tempo) bierzemy tylko z 2026.

**Jak porównujemy uczciwie:** kierowcy mogą odpaść w różnych segmentach (jeden w Q1, drugi w Q3). Porównujemy czasy z **ostatniego segmentu, w którym jechali obaj** — wtedy warunki toru były takie same.

In [9]:
def teammate_gaps(q):
    """Dla jednej sesji kwalifikacji: różnica % do partnera z ostatniego wspólnego segmentu (Q1/Q2/Q3)."""
    rows = []
    for team, g in q.groupby("team_name"):
        if len(g) != 2:
            continue
        a, b = g.iloc[0], g.iloc[1]
        if not (isinstance(a["duration"], list) and isinstance(b["duration"], list)):
            continue
        common = [i for i in range(3) if a["duration"][i] and b["duration"][i]]
        if not common:
            continue
        k = common[-1]
        gap = (a["duration"][k] / b["duration"][k] - 1) * 100
        rows.append({"name_acronym": a["name_acronym"], "teammate": b["name_acronym"], "gap_pct": gap})
        rows.append({"name_acronym": b["name_acronym"], "teammate": a["name_acronym"], "gap_pct": -gap})
    return pd.DataFrame(rows)

frames = []
for year in [2023, 2024, 2025]:
    q_sessions = get("sessions", year=year, session_name="Qualifying")
    for s in q_sessions.itertuples():
        res = get("session_result", session_key=s.session_key)
        if res.empty:
            continue
        df = res.merge(weekend_drivers(s.meeting_key), on="driver_number", how="left")
        gaps = teammate_gaps(df)
        gaps["year"] = year
        gaps["meeting_key"] = s.meeting_key
        frames.append(gaps)
    print(year, "gotowe")

teammate_hist = pd.concat(frames, ignore_index=True)
print(teammate_hist.shape)
teammate_hist.head()

2023 gotowe
2024 gotowe
2025 gotowe
(1370, 5)


,name_acronym,teammate,gap_pct,year,meeting_key
0,BOT,ZHO,-0.032797,2023,1141
1,ZHO,BOT,0.032797,2023,1141
2,TSU,DEV,-0.782666,2023,1141
3,DEV,TSU,0.782666,2023,1141
4,OCO,GAS,-0.730085,2023,1141


**Co tu się dzieje:**
- Dla każdego zespołu z dokładnie dwoma kierowcami bierzemy listy `[Q1, Q2, Q3]` i szukamy segmentów, w których **obaj** mają czas (`common`). `k = common[-1]` to ostatni z nich.
- Każde porównanie zapisujemy dwa razy — z perspektywy każdego kierowcy (raz `+gap`, raz `-gap`). Ujemne = szybszy od partnera.
- `isinstance(..., list)` chroni przed kierowcą, który w ogóle nie wyjechał (wtedy `duration` to `None`).
- To najdłuższa komórka przy pierwszym uruchomieniu (~140 zapytań) — potem czyta z dysku.

**Ograniczenie, o którym trzeba pamiętać:** to miara **względem partnera**. Kierowca obok mistrza świata wyjdzie gorzej niż ten sam kierowca obok słabego partnera. Na pierwszą wersję wystarczy.

## 1.9 Historia Singapuru (2023–2025)

**Co robimy:** pobieramy kwalifikacje i wyścigi z trzech ostatnich GP Singapuru.

**Dlaczego:** w EDA sprawdzimy, czy Singapur to tor, na którym pozycja startowa przesądza o wyniku jeszcze bardziej niż średnio. Samochody były inne, ale ulice te same.

In [10]:
frames = []
for year in [2023, 2024, 2025]:
    sg = get("sessions", year=year, country_name="Singapore")
    meeting = sg["meeting_key"].iloc[0]
    q_key = sg.loc[sg["session_name"] == "Qualifying", "session_key"].item()
    r_key = sg.loc[sg["session_name"] == "Race", "session_key"].item()
    q = get("session_result", session_key=q_key)[["driver_number", "position"]].rename(columns={"position": "quali_pos"})
    r = get("session_result", session_key=r_key)[["driver_number", "position", "dnf"]].rename(columns={"position": "race_pos"})
    df = r.merge(q, on="driver_number", how="left").merge(weekend_drivers(meeting), on="driver_number", how="left")
    df["year"] = year
    frames.append(df)

singapore_hist = pd.concat(frames, ignore_index=True)
singapore_hist[singapore_hist["race_pos"] <= 3].sort_values(["year", "race_pos"])

,driver_number,race_pos,dnf,quali_pos,name_acronym,full_name,team_name,year
0,55,1.0,False,1.0,SAI,Carlos SAINZ,Ferrari,2023
1,4,2.0,False,4.0,NOR,Lando NORRIS,McLaren,2023
2,44,3.0,False,5.0,HAM,Lewis HAMILTON,Mercedes,2023
19,4,1.0,False,1.0,NOR,Lando NORRIS,McLaren,2024
20,1,2.0,False,2.0,VER,Max VERSTAPPEN,Red Bull Racing,2024
21,81,3.0,False,5.0,PIA,Oscar PIASTRI,McLaren,2024
39,63,1.0,False,1.0,RUS,George RUSSELL,Mercedes,2025
40,1,2.0,False,2.0,VER,Max VERSTAPPEN,Red Bull Racing,2025
41,4,3.0,False,5.0,NOR,Lando NORRIS,McLaren,2025


**Co tu się dzieje:**
- `sessions?year=...&country_name=Singapore` — wszystkie sesje weekendu w Singapurze danego roku; z nich wyciągamy klucze kwalifikacji i wyścigu.
- Łączymy wynik wyścigu z wynikiem kwalifikacji po `driver_number` (w obrębie jednego weekendu numer jest jednoznaczny) i doklejamy nazwiska.
- Na wyjściu podium każdego roku z pozycją startową — pierwsza odpowiedź na pytanie „czy w Singapurze wygrywa się z przodu?”.

## 1.10 Zapis

**Co robimy:** zapisujemy wszystkie tabele do CSV.

**Dlaczego:** `02_eda`, `03_model` i `04_prediction` zaczynają od wczytania tych plików — bez dotykania API. Kolumnę `duration` z kwalifikacji (lista) pomijamy, bo CSV nie przechowuje list; potrzebne liczby już z niej wyciągnęliśmy.

In [11]:
Path("data").mkdir(exist_ok=True)
races.to_csv("data/races_2026.csv", index=False)
results.to_csv("data/results_2026.csv", index=False)
quali.drop(columns=["duration", "gap_to_leader"]).to_csv("data/quali_2026.csv", index=False)
sprint.to_csv("data/sprint_2026.csv", index=False)
fastest_laps.to_csv("data/fastest_laps_2026.csv", index=False)
teammate_hist.to_csv("data/teammate_gaps_hist.csv", index=False)
singapore_hist.to_csv("data/singapore_hist.csv", index=False)
print("Zapisano:", sorted(p.name for p in Path("data").glob("*.csv")))

Zapisano: ['fastest_laps_2026.csv', 'features.csv', 'prediction_singapore_2026.csv', 'quali_2026.csv', 'races_2026.csv', 'results_2026.csv', 'singapore_hist.csv', 'sprint_2026.csv', 'teammate_gaps_hist.csv']
